# Text Classification with Logistic Regression

In this notebook, we will build a **sentiment classifier** for movie reviews using **logistic regression from scratch**. By the end, you will understand:
1. How raw text becomes a numerical feature vector
2. The sigmoid function and how it produces probabilities
3. Cross-entropy loss and why it measures "how wrong" the model is
4. Stochastic Gradient Descent (SGD) and how weights are learned

**Dataset:** IMDb Movie Reviews (1,000 positive / 1,000 negative)



In [1]:
import nltk
import numpy as np
import random
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import (
    confusion_matrix, accuracy_score,
    precision_score, recall_score, f1_score
)

nltk.download('movie_reviews', quiet=True)
from nltk.corpus import movie_reviews

print("Libraries loaded.")

Libraries loaded.


In [2]:
# Load and shuffle the IMDb dataset
documents = [(list(movie_reviews.words(fileid)), category)
             for category in movie_reviews.categories()
             for fileid in movie_reviews.fileids(category)]

random.seed(42)
random.shuffle(documents)

texts  = [" ".join(words) for words, _ in documents]
labels = np.array([1 if cat == 'pos' else 0 for _, cat in documents])

print(f"Total documents: {len(texts)}")
print(f"Positive reviews: {labels.sum()}, Negative reviews: {len(labels) - labels.sum()}")

Total documents: 2000
Positive reviews: 1000, Negative reviews: 1000


In [3]:
# 80/20 train-test split
X_train, X_test, y_train, y_test = train_test_split(
    texts, labels, test_size=0.2, random_state=42
)

# Bag-of-Words: map each word to a column index (top 5000 words)
vectorizer = CountVectorizer(max_features=5000, stop_words='english')
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec  = vectorizer.transform(X_test)

print(f"Training set size: {len(X_train)}")
print(f"Test set size:     {len(X_test)}")
print(f"Feature matrix shape (train): {X_train_vec.shape}")
print(f"Feature matrix shape (test):  {X_test_vec.shape}")

Training set size: 1600
Test set size:     400
Feature matrix shape (train): (1600, 5000)
Feature matrix shape (test):  (400, 5000)


### What just happened?
Each row is a document. Each column is a word from our vocabulary (top 5,000).
The value at `[i, j]` is the **count** of word `j` in document `i`.

Let's look at a few word indices to build intuition:

In [4]:
# Inspect the vocabulary
vocab = vectorizer.get_feature_names_out()
print("First 10 words in vocabulary:", vocab[:10])
print()

# Check the vector for one specific review
sample_idx = 0
sample_vector = X_train_vec[sample_idx].toarray()[0]
non_zero_indices = np.where(sample_vector > 0)[0]
print(f"Review 0 (first 10 non-zero features):")
for idx in non_zero_indices[:10]:
    print(f"  '{vocab[idx]}' appears {int(sample_vector[idx])} times")

First 10 words in vocabulary: ['000' '10' '100' '101' '11' '12' '13' '13th' '14' '15']

Review 0 (first 10 non-zero features):
  'actresses' appears 1 times
  'actually' appears 1 times
  'alien' appears 1 times
  'america' appears 1 times
  'angeles' appears 1 times
  'answer' appears 1 times
  'appealing' appears 1 times
  'attitude' appears 1 times
  'best' appears 1 times
  'better' appears 2 times


### TODO 1: Implement the Sigmoid Function

The sigmoid function maps any real number `z` to a probability between 0 and 1:

$$ \sigma(z) = \frac{1}{1 + e^{-z}} $$

**Your task:** Implement `sigmoid(z)` below.

**Hints:**
- Use `np.exp(-z)` for the exponential
- Clip `z` to `[-500, 500]` before computing to avoid overflow warnings

In [5]:
def sigmoid(z):
    """Return sigmoid(z), preserving the shape of a scalar or array."""
    z = np.clip(z, -500, 500)  # avoid overflow in exp
    return 1.0 / (1.0 + np.exp(-z))

In [6]:
# --- UNIT TESTS FOR TODO 1 ---
assert round(float(sigmoid(0)), 4) == 0.5,       "sigmoid(0) should be 0.5"
assert round(float(sigmoid(2)), 4) == 0.8808,    "sigmoid(2) should be 0.8808"
assert round(float(sigmoid(-2)), 4) == 0.1192,   "sigmoid(-2) should be 0.1192"
assert round(float(sigmoid(10)), 4) == 1.0,      "sigmoid(10) should be ~1.0"

print("TODO 1 passed: sigmoid is correct!")

TODO 1 passed: sigmoid is correct!


### TODO 2: Implement Cross-Entropy Loss

The document defines the loss for a single observation:

$$ L_{CE}(\hat{y}, y) = -\left[ y \log(\hat{y}) + (1 - y) \log(1 - \hat{y}) \right] $$

**Why this formula?**
- If `y = 1` (true label is positive): loss = `-log(ŷ)` → small if ŷ is close to 1
- If `y = 0` (true label is negative): loss = `-log(1 - ŷ)` → small if ŷ is close to 0

**Your task:** Implement this formula.

**Hint:** Add `1e-15` to `y_hat` and `1 - y_hat` to avoid `log(0)`.

In [7]:
def cross_entropy_loss(y_hat, y):
    """Binary cross-entropy loss for one example."""
    eps = 1e-15
    y_hat = np.clip(y_hat, eps, 1.0 - eps)  # avoid log(0)
    return float(-(y * np.log(y_hat) + (1 - y) * np.log(1 - y_hat)))

### TODO 3: Implement `fit()` (Gradient Descent)
### TODO 4: Implement `predict_proba()`

The class below is a **scaffold**. Fill in the two TODOs.

**Gradient descent update (per sample):**
$$ w_j \leftarrow w_j - \eta \cdot (\hat{y} - y) \cdot x_j $$
$$ b \leftarrow b - \eta \cdot (\hat{y} - y) $$

**Hint:** Remember to convert `X` from a **sparse matrix** to dense with `.toarray()` before doing element-wise math.

In [8]:
class CustomLogisticRegression:
    def __init__(self, learning_rate=0.01, epochs=10):
        self.lr = learning_rate
        self.epochs = epochs
        self.w = None
        self.b = 0.0

    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.w = np.zeros(n_features, dtype=float)
        self.b = 0.0
        X_dense = X.toarray() if hasattr(X, "toarray") else np.asarray(X)
        y = np.asarray(y)

        for epoch in range(self.epochs):
            for i in range(n_samples):
                xi = X_dense[i]
                yi = y[i]
                linear_model = np.dot(xi, self.w) + self.b
                y_pred = sigmoid(linear_model)

                # Gradient of binary cross-entropy for this sample
                error = y_pred - yi
                dw = error * xi
                db = error
                self.w -= self.lr * dw
                self.b -= self.lr * db
        return self

    def predict_proba(self, X):
        X_dense = X.toarray() if hasattr(X, "toarray") else np.asarray(X)
        z = np.dot(X_dense, self.w) + self.b
        return sigmoid(z)

    def predict(self, X):
        y_proba = self.predict_proba(X)
        return (y_proba > 0.5).astype(int)

In [9]:
# --- SANITY CHECK FOR TODOS 3 & 4 ---
# Train on a tiny sample, then test the output shape and range
X_small = X_train_vec[:50]
y_small = y_train[:50]
model_check = CustomLogisticRegression(learning_rate=0.1, epochs=1)
model_check.fit(X_small, y_small)
proba_check = model_check.predict_proba(X_train_vec[:10])
assert proba_check.shape == (10,), f"Wrong shape: {proba_check.shape}"
assert np.all(proba_check >= 0) and np.all(proba_check <= 1), "Probabilities out of [0,1]!"
print(" TODOs 3 & 4 passed: model trains and outputs valid probabilities.")

 TODOs 3 & 4 passed: model trains and outputs valid probabilities.


In [10]:
print("Training Custom Logistic Regression...")
custom_lr = CustomLogisticRegression(learning_rate=0.1, epochs=10)
custom_lr.fit(X_train_vec, y_train)
print("Training complete!")

Training Custom Logistic Regression...
Training complete!


Evaluate the Model

In [11]:
y_pred_custom = custom_lr.predict(X_test_vec)

print("--- Custom Logistic Regression Evaluation ---")
print(f"Accuracy:  {accuracy_score(y_test, y_pred_custom):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_custom):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred_custom):.4f}")
print(f"F1-Score:  {f1_score(y_test, y_pred_custom):.4f}")
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_custom))

--- Custom Logistic Regression Evaluation ---
Accuracy:  0.8075
Precision: 0.7793
Recall:    0.8607
F1-Score:  0.8180

Confusion Matrix:
[[150  49]
 [ 28 173]]


Compare with Scikit-Learn

In [12]:
from sklearn.linear_model import SGDClassifier

sklearn_lr = SGDClassifier(loss='log_loss', max_iter=1000, random_state=42)
sklearn_lr.fit(X_train_vec, y_train)
y_pred_sklearn = sklearn_lr.predict(X_test_vec)

print("--- Scikit-Learn Logistic Regression Evaluation ---")
print(f"Accuracy:  {accuracy_score(y_test, y_pred_sklearn):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_sklearn):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred_sklearn):.4f}")
print(f"F1-Score:  {f1_score(y_test, y_pred_sklearn):.4f}")

--- Scikit-Learn Logistic Regression Evaluation ---
Accuracy:  0.8025
Precision: 0.7961
Recall:    0.8159
F1-Score:  0.8059


### Discussion: Custom model vs. Scikit-Learn

The custom model is not guaranteed to beat Scikit-Learn. Scikit-Learn's `SGDClassifier` uses a highly optimized implementation and exposes additional training options. Results can differ because of initialization, update strategy, regularization, learning-rate schedule, convergence criteria, and number of iterations.

`SGDClassifier` exposes (among others):
- `alpha`: regularization strength.
- `penalty`: regularization type (`l2`, `l1`, `elasticnet`, or `None`).
- `learning_rate` and `eta0`: learning-rate schedule and initial rate.
- `max_iter` and `tol`: maximum epochs and stopping tolerance.
- `shuffle`: whether to shuffle samples between epochs.
- `early_stopping`, `validation_fraction`, and `n_iter_no_change`: early-stopping controls.
- `average`: whether to use averaged SGD weights.

The custom implementation hardcodes several of these choices and uses a simple fixed learning rate with one update per sample.

Deployment Function

In [13]:
def deploy_sentiment_model(text, model, vectorizer):
    """Predict sentiment for a raw text string."""
    text_vec = vectorizer.transform([text])
    pred = model.predict(text_vec)[0]
    proba = model.predict_proba(text_vec)[0]
    sentiment = "Positive" if pred == 1 else "Negative"
    confidence = proba[pred]
    return sentiment, confidence

# Test on the review from the slides
new_review = ("It's hokey. There are virtually no surprises, and the writing is "
              "second-rate. So why was it so enjoyable? For one thing, the cast "
              "is great.")
prediction, prob = deploy_sentiment_model(new_review, sklearn_lr, vectorizer)
print(f"Review: {new_review}")
print(f"Prediction: {prediction} (Probability: {prob:.4f})")

Review: It's hokey. There are virtually no surprises, and the writing is second-rate. So why was it so enjoyable? For one thing, the cast is great.
Prediction: Positive (Probability: 1.0000)


### Challenge 1: Decision Boundary

A threshold of **0.3** labels more reviews as positive than a threshold of 0.5.
- **Recall usually increases** because more actual positive reviews are captured.
- **Precision usually decreases** because more negative reviews may be incorrectly labeled positive.
- For a spam filter, a higher threshold is often safer when the cost of sending a legitimate email to spam is high. It reduces false positives, although it may miss some spam. The best threshold depends on which error is more costly.

### Challenge 2: L2 Regularization

L2 regularization adds `lambda * w` to the weight gradient. It discourages very large weights and can reduce overfitting. Whether test accuracy improves is an empirical question; it is not guaranteed.

### Challenge 3: Mini-Batch Gradient Descent

Mini-batch gradient descent computes one average gradient for each group of 32 samples. It performs fewer parameter updates than SGD, and uses vectorized NumPy operations, which often makes training faster. The F1 score may improve or worsen depending on the learning rate, batch size, and data; compare the measured result below.

In [14]:
# Challenge 1: evaluate the custom model with a 0.3 threshold
y_proba_custom = custom_lr.predict_proba(X_test_vec)
y_pred_03 = (y_proba_custom >= 0.3).astype(int)
print('Threshold = 0.30')
print(f'Accuracy:  {accuracy_score(y_test, y_pred_03):.4f}')
print(f'Precision: {precision_score(y_test, y_pred_03, zero_division=0):.4f}')
print(f'Recall:    {recall_score(y_test, y_pred_03, zero_division=0):.4f}')
print(f'F1-Score:  {f1_score(y_test, y_pred_03, zero_division=0):.4f}')
print('Confusion matrix:\n', confusion_matrix(y_test, y_pred_03))

Threshold = 0.30
Accuracy:  0.8075
Precision: 0.7743
Recall:    0.8706
F1-Score:  0.8197
Confusion matrix:
 [[148  51]
 [ 26 175]]


In [15]:
# Challenge 2: logistic regression with L2 regularization
class L2LogisticRegression(CustomLogisticRegression):
    def __init__(self, learning_rate=0.01, epochs=10, lambda_=0.01):
        super().__init__(learning_rate, epochs)
        self.lambda_ = lambda_

    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.w = np.zeros(n_features, dtype=float)
        self.b = 0.0
        X_dense = X.toarray() if hasattr(X, 'toarray') else np.asarray(X)
        y = np.asarray(y)
        for epoch in range(self.epochs):
            for i in range(n_samples):
                xi, yi = X_dense[i], y[i]
                y_pred = sigmoid(np.dot(xi, self.w) + self.b)
                error = y_pred - yi
                dw = error * xi + self.lambda_ * self.w
                db = error  # do not regularize the bias
                self.w -= self.lr * dw
                self.b -= self.lr * db
        return self

l2_model = L2LogisticRegression(learning_rate=0.1, epochs=10, lambda_=0.01)
l2_model.fit(X_train_vec, y_train)
y_pred_l2 = l2_model.predict(X_test_vec)
print('L2 regularized model (lambda = 0.01)')
print(f'Accuracy: {accuracy_score(y_test, y_pred_l2):.4f}')
print(f'Precision: {precision_score(y_test, y_pred_l2, zero_division=0):.4f}')
print(f'Recall: {recall_score(y_test, y_pred_l2, zero_division=0):.4f}')
print(f'F1-Score: {f1_score(y_test, y_pred_l2, zero_division=0):.4f}')
print('Compare this accuracy with the unregularized custom model above.')

L2 regularized model (lambda = 0.01)
Accuracy: 0.7675
Precision: 0.7477
Recall: 0.8109
F1-Score: 0.7780
Compare this accuracy with the unregularized custom model above.


In [16]:
# Challenge 3: mini-batch gradient descent (batch size = 32)
class MiniBatchLogisticRegression(CustomLogisticRegression):
    def __init__(self, learning_rate=0.01, epochs=10, batch_size=32):
        super().__init__(learning_rate, epochs)
        self.batch_size = batch_size

    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.w = np.zeros(n_features, dtype=float)
        self.b = 0.0
        y = np.asarray(y)
        for epoch in range(self.epochs):
            # Shuffle the sample order at the start of each epoch
            indices = np.random.permutation(n_samples)
            for start in range(0, n_samples, self.batch_size):
                batch_indices = indices[start:start + self.batch_size]
                X_batch = X[batch_indices]
                X_batch = X_batch.toarray() if hasattr(X_batch, 'toarray') else np.asarray(X_batch)
                y_batch = y[batch_indices]
                predictions = sigmoid(X_batch @ self.w + self.b)
                errors = predictions - y_batch
                dw = (X_batch.T @ errors) / len(batch_indices)
                db = np.mean(errors)
                self.w -= self.lr * dw
                self.b -= self.lr * db
        return self

import time
start_time = time.perf_counter()
mini_batch_model = MiniBatchLogisticRegression(learning_rate=0.01, epochs=10, batch_size=32)
mini_batch_model.fit(X_train_vec, y_train)
mini_batch_seconds = time.perf_counter() - start_time
y_pred_batch = mini_batch_model.predict(X_test_vec)
print(f'Mini-batch training time: {mini_batch_seconds:.2f} seconds')
print(f'Accuracy:  {accuracy_score(y_test, y_pred_batch):.4f}')
print(f'Precision: {precision_score(y_test, y_pred_batch, zero_division=0):.4f}')
print(f'Recall:    {recall_score(y_test, y_pred_batch, zero_division=0):.4f}')
print(f'F1-Score:  {f1_score(y_test, y_pred_batch, zero_division=0):.4f}')
print('Compare the F1 score and training time with the original custom model.')

Mini-batch training time: 1.82 seconds
Accuracy:  0.8100
Precision: 0.8019
Recall:    0.8259
F1-Score:  0.8137
Compare the F1 score and training time with the original custom model.


## Takeaways

A linear model on top of a Bag-of-Words representation is a surprisingly strong baseline for
sentiment analysis, and every piece of it fits in a few lines of NumPy.

What actually moves the needle here:

- **Feature choice.** Unigrams + stop-word removal already capture most of the signal; bigrams
  or TF-IDF weighting are cheap next steps.
- **Learning rate and epochs.** Too high and the loss diverges; too low and ten epochs isn't enough.
- **Threshold.** The 0.5 cut-off is a default, not a law — pick it from the cost of each error type.
- **Batching.** Mini-batch updates are both faster and less noisy than one-sample SGD.

### Where to go next

1. Swap `CountVectorizer` for `TfidfVectorizer(min_df=2, ngram_range=(1, 2))` and re-run the comparison.
2. Add a learning-rate schedule and plot the loss per epoch.
3. Try the same pipeline on a multiclass corpus (e.g. 20 Newsgroups) with softmax regression.
